# Time Series & Trend Analysis

This notebook demonstrates advanced T-SQL patterns for time series analysis using the AdventureWorksDW2022 data warehouse.

**Questions covered:**
5. Running Total of Sales by Month
6. Month-over-Month Sales Growth (with Date Spine)
7. 3-Month Moving Average
8. Year-to-Date (YTD) Sales

## Setup

In [ ]:
import urllib
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Database connection - adjust server name as needed
server = 'localhost'
database = 'AdventureWorksDW2022'
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'
conn_url = f'mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(conn_str)}'
engine = create_engine(conn_url)
print('Connected to AdventureWorksDW2022')

---
## Q5: Running Total of Sales by Month

**Business Question:** Calculate a running total of internet sales revenue by month, partitioned by product category.

**Key Techniques:**
- `SUM() OVER (PARTITION BY ... ORDER BY ...)` for cumulative sum
- Default window frame (RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)

In [ ]:
q5_sql = """
WITH MonthlySales AS (
    SELECT 
        dd.CalendarYear,
        dd.MonthNumberOfYear,
        dd.EnglishMonthName AS Month,
        cat.ProductCategoryKey AS CategoryID,
        cat.EnglishProductCategoryName AS Category,
        SUM(fis.SalesAmount) AS MonthlySales
    FROM dbo.FactInternetSales fis
    INNER JOIN dbo.DimDate dd ON fis.OrderDateKey = dd.DateKey
    INNER JOIN dbo.DimProduct p ON fis.ProductKey = p.ProductKey
    INNER JOIN dbo.DimProductSubcategory sub ON p.ProductSubcategoryKey = sub.ProductSubcategoryKey
    INNER JOIN dbo.DimProductCategory cat ON sub.ProductCategoryKey = cat.ProductCategoryKey
    GROUP BY dd.CalendarYear, dd.MonthNumberOfYear, dd.EnglishMonthName, cat.ProductCategoryKey, cat.EnglishProductCategoryName
),
CategoryRunningTotal AS (
    SELECT 
        CalendarYear,
        MonthNumberOfYear,
        Month,
        CategoryID,
        Category,
        MonthlySales,
        SUM(MonthlySales) OVER (PARTITION BY CategoryID ORDER BY CalendarYear, MonthNumberOfYear) AS RunningTotal
    FROM MonthlySales
)
SELECT 
    CalendarYear,
    MonthNumberOfYear,
    Month,
    Category,
    MonthlySales,
    RunningTotal
FROM CategoryRunningTotal
ORDER BY Category, CalendarYear, MonthNumberOfYear
"""

df_q5 = pd.read_sql(q5_sql, engine)
df_q5.head(15)

In [ ]:
# Visualization: Running Total by Category
HIGHLIGHT_COLOR = '#2563eb'
MUTED_COLOR = '#d1d5db'

fig, ax = plt.subplots(figsize=(14, 6))

categories = df_q5['Category'].unique()
top_category = df_q5.groupby('Category')['RunningTotal'].max().idxmax()

for category in categories:
    cat_data = df_q5[df_q5['Category'] == category]
    color = HIGHLIGHT_COLOR if category == top_category else MUTED_COLOR
    linewidth = 2.5 if category == top_category else 1.5
    ax.plot(range(len(cat_data)), cat_data['RunningTotal'] / 1_000_000, 
            label=category, linewidth=linewidth, color=color)

ax.set_xlabel('Time Period')
ax.set_ylabel('Running Total Revenue ($M)')
ax.set_title('Cumulative Revenue by Product Category', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Show fewer x-tick labels
tick_positions = range(0, len(df_q5[df_q5['Category'] == 'Bikes']), 6)
tick_labels = [f"{df_q5[df_q5['Category'] == 'Bikes'].iloc[i]['CalendarYear']}-{df_q5[df_q5['Category'] == 'Bikes'].iloc[i]['MonthNumberOfYear']:02d}" for i in tick_positions]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

plt.tight_layout()
plt.show()

---
## Q6: Month-over-Month Sales Growth (Date Spine Pattern)

**Business Question:** Calculate the percentage change in sales revenue compared to the previous month for each territory, including months with zero sales.

**Key Techniques:**
- **Date Spine** via `CROSS JOIN` to ensure continuous time series
- `LEFT JOIN` to preserve months with no sales
- `COALESCE()` to handle NULL as zero
- `LAG()` for month-over-month comparison

In [ ]:
q6_sql = """
WITH DateSpine AS (
    -- Create all distinct year-month combinations within the sales date range
    SELECT DISTINCT 
        CalendarYear,
        MonthNumberOfYear AS MonthID,
        EnglishMonthName AS Month
    FROM dbo.DimDate
    WHERE DateKey BETWEEN 
        (SELECT MIN(OrderDateKey) FROM dbo.FactInternetSales) AND
        (SELECT MAX(OrderDateKey) FROM dbo.FactInternetSales)
),
TerritoryMonthJoin AS (
    -- Cross join: all Territory x Month combinations
    SELECT 
        ds.CalendarYear,
        ds.MonthID,
        ds.Month,
        st.SalesTerritoryCountry AS Country
    FROM DateSpine ds
    CROSS JOIN (
        SELECT DISTINCT SalesTerritoryCountry
        FROM dbo.DimSalesTerritory
        WHERE SalesTerritoryCountry IS NOT NULL AND SalesTerritoryCountry <> 'NA'
    ) st
),
TerritoryMonthlySales AS (
    -- Left Join to preserve all months, including $0 sales
    SELECT
        tmj.CalendarYear,
        tmj.MonthID,
        tmj.Month,
        tmj.Country,
        COALESCE(sales.MonthlySales, 0) AS MonthlySales
    FROM TerritoryMonthJoin tmj
    LEFT JOIN (
        SELECT 
            st.SalesTerritoryCountry AS Country,
            dd.CalendarYear,
            dd.MonthNumberOfYear AS MonthID,
            SUM(fis.SalesAmount) AS MonthlySales
        FROM dbo.FactInternetSales fis
        INNER JOIN dbo.DimDate dd ON fis.OrderDateKey = dd.DateKey
        INNER JOIN dbo.DimSalesTerritory st ON fis.SalesTerritoryKey = st.SalesTerritoryKey
        GROUP BY st.SalesTerritoryCountry, dd.CalendarYear, dd.MonthNumberOfYear
    ) sales
        ON tmj.Country = sales.Country
        AND tmj.CalendarYear = sales.CalendarYear
        AND tmj.MonthID = sales.MonthID
),
MoMGrowth AS (
    SELECT 
        CalendarYear,
        MonthID,
        Month,
        Country,
        MonthlySales,
        LAG(MonthlySales) OVER (PARTITION BY Country ORDER BY CalendarYear, MonthID) AS PreviousMonthSales,
        MonthlySales - LAG(MonthlySales) OVER (PARTITION BY Country ORDER BY CalendarYear, MonthID) AS MoMChange
    FROM TerritoryMonthlySales
)
SELECT 
    CalendarYear,
    MonthID,
    Month,
    Country,
    MonthlySales,
    PreviousMonthSales,
    MoMChange,
    CASE WHEN PreviousMonthSales = 0 THEN NULL 
         ELSE MoMChange / PreviousMonthSales * 100 END AS MoMGrowthRate
FROM MoMGrowth
ORDER BY Country, CalendarYear, MonthID
"""

df_q6 = pd.read_sql(q6_sql, engine)
df_q6.head(20)

In [ ]:
# Visualization: MoM Growth Heatmap for a single year
year_filter = df_q6['CalendarYear'].max()
df_year = df_q6[df_q6['CalendarYear'] == year_filter]

pivot = df_year.pivot(index='Country', columns='MonthID', values='MoMGrowthRate')
pivot.columns = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'][:len(pivot.columns)]

fig, ax = plt.subplots(figsize=(14, 6))

# Professional blue-grey colormap
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap=cmap, center=0, ax=ax, 
            cbar_kws={'label': 'MoM Growth %'}, linewidths=0.5, linecolor='white')

ax.set_title(f'Month-over-Month Growth Rate by Country ({year_filter})', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Country')
plt.tight_layout()
plt.show()

---
## Q7: 3-Month Moving Average

**Business Question:** Compute a 3-month moving average of sales to smooth out seasonal fluctuations.

**Key Techniques:**
- `AVG() OVER (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` for trailing average
- Window frame specification for precise row boundaries

In [ ]:
q7_sql = """
WITH MonthlySales AS (
    SELECT 
        dd.CalendarYear,
        dd.MonthNumberOfYear AS MonthID,
        dd.EnglishMonthName AS Month,
        SUM(fis.SalesAmount) AS MonthlySales
    FROM dbo.FactInternetSales fis
    INNER JOIN dbo.DimDate dd ON fis.OrderDateKey = dd.DateKey
    GROUP BY dd.CalendarYear, dd.MonthNumberOfYear, dd.EnglishMonthName
),
MovingAverage AS (
    SELECT
        CalendarYear,
        MonthID,
        Month,
        MonthlySales,
        AVG(MonthlySales) OVER (ORDER BY CalendarYear, MonthID ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS MovingAvg3Month,
        COUNT(MonthID) OVER (ORDER BY CalendarYear, MonthID ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS NumOfMonths
    FROM MonthlySales
)
SELECT
    CalendarYear,
    MonthID,
    Month,
    MonthlySales,
    MovingAvg3Month,
    NumOfMonths
FROM MovingAverage
ORDER BY CalendarYear, MonthID
"""

df_q7 = pd.read_sql(q7_sql, engine)
df_q7

In [ ]:
# Visualization: Actual vs 3-Month Moving Average
HIGHLIGHT_COLOR = '#2563eb'
MUTED_COLOR = '#d1d5db'

fig, ax = plt.subplots(figsize=(14, 6))

x = range(len(df_q7))
ax.plot(x, df_q7['MonthlySales'] / 1000, label='Actual Sales', alpha=0.7, linewidth=1.5, color=MUTED_COLOR)
ax.plot(x, df_q7['MovingAvg3Month'] / 1000, label='3-Month Moving Avg', linewidth=2.5, color=HIGHLIGHT_COLOR)

# Highlight where moving average doesn't have full 3 months
incomplete = df_q7[df_q7['NumOfMonths'] < 3]
ax.scatter(incomplete.index, incomplete['MovingAvg3Month'] / 1000, color='#f59e0b', s=50, 
           label='Incomplete Window (<3 months)', zorder=5)

ax.set_xlabel('Time Period')
ax.set_ylabel('Sales ($K)')
ax.set_title('Monthly Sales with 3-Month Moving Average (Smoothing Seasonality)', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# X-axis labels
tick_positions = range(0, len(df_q7), 3)
tick_labels = [f"{df_q7.iloc[i]['CalendarYear']}-{df_q7.iloc[i]['MonthID']:02d}" for i in tick_positions]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

plt.tight_layout()
plt.show()

---
## Q8: Year-to-Date (YTD) Sales

**Business Question:** Calculate YTD sales for each product, resetting at the start of each calendar year.

**Key Techniques:**
- `SUM() OVER (PARTITION BY year, product ORDER BY month)` for YTD accumulation
- Partition boundary resets the running total each year

In [ ]:
q8_sql = """
WITH MonthlySales AS (
    SELECT 
        dd.CalendarYear,
        dd.MonthNumberOfYear AS MonthID,
        dd.EnglishMonthName AS Month,
        p.ProductKey,
        p.EnglishProductName AS Product,
        SUM(fis.SalesAmount) AS MonthlySales
    FROM dbo.FactInternetSales fis
    INNER JOIN dbo.DimDate dd ON fis.OrderDateKey = dd.DateKey
    INNER JOIN dbo.DimProduct p ON fis.ProductKey = p.ProductKey
    GROUP BY dd.CalendarYear, dd.MonthNumberOfYear, dd.EnglishMonthName, p.ProductKey, p.EnglishProductName
),
YTDSales AS (
    SELECT 
        CalendarYear,
        MonthID,
        Month,
        ProductKey,
        Product,
        MonthlySales,
        SUM(MonthlySales) OVER (PARTITION BY CalendarYear, ProductKey ORDER BY MonthID) AS YTDSales
    FROM MonthlySales
)
SELECT
    CalendarYear,
    MonthID,
    Month,
    Product,
    MonthlySales,
    YTDSales
FROM YTDSales
ORDER BY Product, CalendarYear, MonthID
"""

df_q8 = pd.read_sql(q8_sql, engine)
df_q8.head(20)

In [ ]:
# Visualization: YTD Sales for Top 5 Products (Latest Year)
HIGHLIGHT_COLOR = '#2563eb'
MUTED_COLOR = '#d1d5db'

latest_year = df_q8['CalendarYear'].max()
top_products = df_q8[df_q8['CalendarYear'] == latest_year].groupby('Product')['YTDSales'].max().nlargest(5).index.tolist()

fig, ax = plt.subplots(figsize=(12, 6))

top_product = top_products[0]
for product in top_products:
    prod_data = df_q8[(df_q8['Product'] == product) & (df_q8['CalendarYear'] == latest_year)]
    color = HIGHLIGHT_COLOR if product == top_product else MUTED_COLOR
    linewidth = 2.5 if product == top_product else 1.5
    ax.plot(prod_data['MonthID'], prod_data['YTDSales'] / 1000, marker='o', 
            label=product[:30], linewidth=linewidth, color=color)

ax.set_xlabel('Month')
ax.set_ylabel('YTD Sales ($K)')
ax.set_title(f'Year-to-Date Sales Accumulation - Top 5 Products ({latest_year})', fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Show YTD reset across years for one product
HIGHLIGHT_COLOR = '#2563eb'
MUTED_COLOR = '#d1d5db'

sample_product = top_products[0]
sample_data = df_q8[df_q8['Product'] == sample_product]

fig, ax = plt.subplots(figsize=(14, 5))

years = sorted(sample_data['CalendarYear'].unique())
latest_year = max(years)

for year in years:
    year_data = sample_data[sample_data['CalendarYear'] == year]
    color = HIGHLIGHT_COLOR if year == latest_year else MUTED_COLOR
    linewidth = 2.5 if year == latest_year else 1.5
    ax.plot(year_data['MonthID'], year_data['YTDSales'] / 1000, marker='o', 
            label=str(year), linewidth=linewidth, color=color)

ax.set_xlabel('Month')
ax.set_ylabel('YTD Sales ($K)')
ax.set_title(f'YTD Sales Reset Each Year: {sample_product[:50]}', fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend(title='Year')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## Cleanup

In [ ]:
engine.dispose()
print('Connection closed')